In [ ]:
# run this notebook after 28_methods_smoking_python
# use this notebook to generate files necessary (in aou) to test if the sibling carrying the disruptive
# mutation at rev1 or lig1 has a significantly higher mutation rate compared to the sibling not carrying 
# the mutation, which would suggest clonal hematopoiesis instead of germline
# after this, run 30_rev1_lig1_replication_cpg_R

In [ ]:
missense = fread("./aou_sibs_repair_gene_missense_96_COSMIC.txt")
lof = fread("./aou_sibs_repair_gene_lof_96_COSMIC.txt")
# subset to missense and lof carriers 
missense_rev1 = missense %>% filter(REV1 >= 1)
missense_lig1 = missense %>% filter(LIG1 >= 1)
lof_rev1 = lof %>% filter(REV1 >= 1)
lof_lig1 = lof %>% filter(LIG1 >= 1)

In [ ]:
gt = fread("./vat_repair_genes_missense_lof_trio_parents_siblings_nonref_GT.tsv", sep = "\t", header=TRUE)
gt = as.data.frame(gt)

In [ ]:
variants = fread("./repair_genes_lof_missense_variants.txt", header=TRUE)
variants$type = ifelse(variants$lof, "lof", "missense")
variants$type = ifelse(variants$missense, "missense", "lof")
variants$alleles = paste0('["', variants$ref_allele, '","', variants$alt_allele, '"]')
variants$locus = paste(variants$contig, variants$position, sep =":")

In [ ]:
families = fread("./families_COSMIC_96_mutation_rates.txt") %>% select(fid, ids,
                                                                                            cpg_denominators,
                                                                                            denominators)

cosmic_96 = fread("./sibs_trios_clean_COSMIC_96.txt")
cosmic_types = names(cosmic_96)[2:97]
cpg_names = cosmic_types[which(substr(cosmic_types, 3, 7) == "C>T]G")]
ibd2_sib = fread("./aou_ibd2_giab_removed.tsv")
names(ibd2_sib) = c("id", "denominator")
ibd2_sib$denominator= 4 * ibd2_sib$denominator
cosmic_96 = merge(cosmic_96, ibd2_sib, by = "id", all.x = TRUE)
cosmic_96 = unique(cosmic_96)
cosmic_96$denominator[which(is.na(cosmic_96$denominator))] = 2 * 2294489308
# generate c_a mutation rates 
genome = fread("./hg38.chrom.sizes")
genome = genome %>% filter(V1 %in% paste("chr", 1:22, sep = ""))
genome = sum(genome$V2)
c_a = 907147366/2294489308
c_g = c_a
cpg = 39815717/2294489308
c_t = c_a - cpg
t_c = 1 - c_a
t_g = t_c
t_a = t_c

cosmic_96$c_a_denominator = cosmic_96$denominator * c_a
cosmic_96$c_t_denominator = cosmic_96$denominator * c_t 
cosmic_96$c_g_denominator = cosmic_96$denominator * c_g 

cosmic_96$t_a_denominator = cosmic_96$denominator * t_a
cosmic_96$t_c_denominator = cosmic_96$denominator * t_c 
cosmic_96$t_g_denominator = cosmic_96$denominator * t_g 

cosmic_96$cpg_denominator = cosmic_96$denominator * cpg 

sibling_mutations = fread("./sibs_snps_QC_GIAB_distance_AF_outliers_COSMIC.txt")
sibling_mutations$pair = paste(sibling_mutations$idv1, sibling_mutations$idv2, sep = "_")

unique_individuals = c(unlist(missense_rev1$sibs), unlist(missense_lig1$sibs), unlist(lof_rev1$sibs),
                      unlist(lof_lig1$sibs))
# unique individuals and the pairs they're in 
all_pairs = unique(sibling_mutations %>% select(idv1, idv2, pair))
sib_pairs = all_pairs %>% filter(idv1 %in% unique_individuals | idv2 %in% unique_individuals)
# add denominators 
denom = cosmic_96 %>% select(id, denominator, cpg_denominator)
names(denom)[1] = c("pair")
sib_pairs = merge(sib_pairs, denom, by = "pair") %>% mutate(denominator = denominator/2, 
                                                           cpg_denominator = cpg_denominator/2)

sib_pairs$idv1_rate = NA
sib_pairs$idv2_rate = NA
sib_pairs$idv1_cpg_rate = NA
sib_pairs$idv2_cpg_rate = NA
for(i in 1:nrow(sib_pairs)){
    sub =  sibling_mutations %>% filter(pair == sib_pairs$pair[i])
    sib_pairs$idv1_rate[i] = sum(sub$idv == sib_pairs$idv1[i])/sib_pairs$denominator[i]
    sib_pairs$idv2_rate[i] = sum(sub$idv == sib_pairs$idv2[i])/sib_pairs$denominator[i]
    sib_pairs$idv1_cpg_rate[i] = sum(sub$idv == sib_pairs$idv1[i] & sub$Type %in% cpg_names)/sib_pairs$cpg_denominator[i]*2
    sib_pairs$idv2_cpg_rate[i] = sum(sub$idv == sib_pairs$idv2[i] & sub$Type %in% cpg_names)/sib_pairs$cpg_denominator[i]*2
}

individual_rates = data.frame(idv = unique_individuals, tot_rate = NA, cpg_rate = NA)
for (i in 1:nrow(individual_rates)){
    sib_sub_1 = sib_pairs %>% filter(idv1 == individual_rates$idv[i])
    sib_sub_2 = sib_pairs %>% filter(idv2 == individual_rates$idv[i])
    rates = c()
    cpg_rates = c()
    if (nrow(sib_sub_1) != 0){
        rates = c(rates, sib_sub_1$idv1_rate)
        cpg_rates = c(cpg_rates, sib_sub_1$idv1_cpg_rate)
    }
    if (nrow(sib_sub_2) != 0){
        rates = c(rates, sib_sub_2$idv2_rate)
        cpg_rates = c(cpg_rates, sib_sub_2$idv2_cpg_rate)
    }
    individual_rates$tot_rate[i] = mean(rates)
    individual_rates$cpg_rate[i] = mean(cpg_rates)
}

In [ ]:
add_sibs_columns = function(df){
    df$sibs = vector("list", nrow(df))
    for (i in 1:nrow(df)){
        df$sibs[[i]] = sapply(df$ids[i], function(x) unique(unlist(strsplit(strsplit(x, split = "\\|")[[1]], "_"))))
    }
    return(df)                 
}

add_sib_carrier = function(df, gene, t){
    df$sib_carrier = vector("list", nrow(df))
    df$compare = NA
    df$num_sibs = NA
    # subset variants 
    variants_sub = variants %>% filter(gene_symbol == gene & type == t)
    # subset gt to these variants 
    gt_sub = merge(variants_sub, gt, by = c("locus", "alleles"), all.x =TRUE)
    for (i in 1:nrow(df)){
        sibs = unlist(df$sibs[i])
        df$num_sibs[i] = length(sibs)
        gt_sibs = gt_sub[,..sibs]
        carriers = c()
        for (j in 1:ncol(gt_sibs)){
            if (any(gt_sibs[,..j][[1]] %in% c("0/1", "1/0", "1|0", "0|1", "1/1", "1|1"))){
                carriers = c(carriers, names(gt_sibs)[j])
            }
        }
        df$sib_carrier[[i]] = carriers
        df$compare[i] = length(carriers) < length(sibs)
    }
    return(df)
}


add_rates = function(df){
    df = df%>% filter(compare)
    df$carrier_rates = vector("list", nrow(df))
    df$non_carrier_rates= vector("list", nrow(df))
    df$carrier_cpg_rates = vector("list", nrow(df))
    df$non_carrier_cpg_rates = vector("list", nrow(df))
    for (i in 1:nrow(df)){
        carriers = df$sib_carrier[[i]]
        non_carriers = df$sibs[[i]][which(!df$sibs[[i]] %in% carriers)]
        df$carrier_rates[[i]] = individual_rates$tot_rate[which(individual_rates$idv %in% carriers)]
        df$non_carrier_rates[[i]] = individual_rates$tot_rate[which(individual_rates$idv %in% non_carriers)]
        df$carrier_cpg_rates[[i]] = individual_rates$cpg_rate[which(individual_rates$idv %in% carriers)]
        df$non_carrier_cpg_rates[[i]] = individual_rates$cpg_rate[which(individual_rates$idv %in% non_carriers)]
    }
    return(df %>% select(fid, carrier_rates, non_carrier_rates, carrier_cpg_rates, non_carrier_cpg_rates))
}                           

In [ ]:
missense_rev1 = add_sib_carrier(missense_rev1 %>% add_sibs_columns(), t = "missense", gene = "REV1") %>% add_rates() 
missense_lig1 = add_sib_carrier(missense_lig1 %>% add_sibs_columns(), t = "missense", gene = "LIG1") %>% add_rates()
lof_rev1 = add_sib_carrier(lof_rev1 %>% add_sibs_columns(), t = "lof", gene = "REV1") %>% add_rates()
lof_lig1 = add_sib_carrier(lof_lig1 %>% add_sibs_columns(), t = "lof", gene = "LIG1") %>% add_rates()


In [ ]:
missense_rev1$type = "missense"
missense_rev1$gene = "REV1"
missense_lig1$type = "missense"
missense_lig1$gene = "LIG1"

lof_rev1$type = "lof"
lof_rev1$gene = "REV1"
lof_lig1$type = "lof"
lof_lig1$gene = "LIG1"
rev1_lig1 = rbind(missense_rev1, missense_lig1) %>% rbind(lof_rev1) %>% rbind(lof_lig1)

In [ ]:
fwrite(rev1_lig1, "rev1_lig1_rates_carrier.txt", sep = "\t", quote = FALSE, row.names = FALSE)